# Fase 2: Creación, Entrenamiento y Evaluación del Modelo CNN

## 1. Carga de los Datos Preprocesados
Comenzamos cargando los tensores unificados que generamos en la Fase 1. Gracias a nuestro procesamiento previo con OpenCV, estos datos ya integran las imágenes de nuestros datasets, estandarizadas a 64x64 píxeles en escala de grises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_train = np.load('../dataset/datos_procesados/X_train.npy')
X_val   = np.load('../dataset/datos_procesados/X_val.npy')
X_test  = np.load('../dataset/datos_procesados/X_test.npy')
Y_train = np.load('../dataset/datos_procesados/Y_train.npy')
Y_val   = np.load('../dataset/datos_procesados/Y_val.npy')
Y_test  = np.load('../dataset/datos_procesados/Y_test.npy')

print(f"Dimensiones de entrenamiento: {X_train.shape}")
print(f"Dimensiones de validación: {X_val.shape}")
print(f"Dimensiones de prueba: {X_test.shape}")

## 2. Definición de la Arquitectura del Modelo (CNN)

Para este problema de clasificación de imágenes, utilizaremos una **Red Neuronal Convolucional (CNN)**. 

### Justificación de las Capas:
* **Conv2D (Capas Convolucionales):** Son el núcleo del modelo. Detectan características espaciales como bordes y formas de los dedos.
* **MaxPooling2D:** Reduce la dimensionalidad de la imagen, conservando las características más importantes detectadas por la convolución y evitando el sobreajuste.
* **Flatten:** Aplana la matriz bidimensional resultante en un vector unidimensional.
* **Dense (Capa Oculta - ReLU):** Capa con función de activación ReLU que introduce no-linealidad y evita el problema del desvanecimiento del gradiente.
* **Dropout & L2 (Regularización):** Apagamos aleatoriamente un 50% de las neuronas en cada iteración y penalizamos los pesos grandes (L2) para evitar el *Overfitting*.
* **Dense (Capa de Salida - Softmax):** Tiene 10 neuronas (una para cada dígito) que generan una distribución de probabilidad.

### Búsqueda de Hiperparámetros y Complejidad
Antes de llegar a la arquitectura final presentada a continuación, se realizaron pruebas con modelos de diferente complejidad (por ejemplo, con solo 2 bloques convolucionales y sin regularización L2). Esto resultó en un claro sobreajuste. Posteriormente se probó modificar el Dropout a 0.2, pero la red actual (con 3 bloques, L2=0.001 y Dropout=0.5) fue la que entregó el mejor balance de pérdida en validación sin sobreajuste, demostrando ser la estructura más adecuada.

In [ ]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 1)))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

# Aplanamiento y Clasificación
model.add(Flatten())
model.add(Dense(128, activation='relu', kernel_regularizer=l2(0.001))) # Regularización L2
model.add(Dropout(0.5)) # Regularización Dropout (Apaga el 50% de neuronas)
model.add(Dense(10, activation='softmax')) # Salida de 10 clases

model.summary()

## 3. Configuración y Callbacks (Early Stopping y TensorBoard)
Utilizaremos el optimizador Adam por su adaptabilidad y rápida convergencia en modelos profundos, y la pérdida *Categorical Crossentropy* debido a que es un problema de clasificación multiclase con etiquetas en formato one-hot. Además, implementaremos **Early Stopping** (Punto 6 del taller) para detener el entrenamiento si el modelo deja de mejorar, y **TensorBoard** para monitoreo.

In [ ]:
import os
import datetime
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

optimizador_estable = Adam(learning_rate=0.0005)

model.compile(optimizer=optimizador_estable, loss='categorical_crossentropy', metrics=['accuracy'])

# Configurar TensorBoard (guardará los logs en la carpeta principal)
log_dir = os.path.join("../logs", "fit", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True,
    verbose=1
)

## 4. Entrenamiento del Modelo con Pesos Balanceados (Class Weights)
Dado que fusionamos múltiples datasets, calculamos dinámicamente los pesos de cada clase para obligar a la red a "prestarle más atención" a los dígitos que tienen menos fotografías, evitando sesgos.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 1. Calcular los pesos de cada clase a partir de las etiquetas de entrenamiento
y_train_labels = np.argmax(Y_train, axis=1)  # convertir de one-hot a etiquetas enteras

pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weights_dict = dict(enumerate(pesos))

print("Pesos calculados por clase:", class_weights_dict)

# 2. Generador de imágenes (Data Augmentation)
datagen = ImageDataGenerator(
    rotation_range=5,
    zoom_range=0.05,
    width_shift_range=0.05,
    height_shift_range=0.05
)

batch_size = 32
epochs = 50

print("Iniciando entrenamiento estabilizado...")

history = model.fit(
    datagen.flow(X_train, Y_train, batch_size=batch_size),
    validation_data=(X_val, Y_val),
    epochs=epochs,
    class_weight=class_weights_dict,   # <- aquí se aplican los pesos
    callbacks=[early_stop, tensorboard_callback],
    verbose=1
)

## 5. Evaluación y Análisis de Resultados
Graficamos las curvas de aprendizaje y evaluamos con los datos de prueba (Test Data) que el modelo jamás ha visto.

In [ ]:
# Gráfica de Precisión y Pérdida
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Precisión')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Pérdida')
plt.show()

# Evaluación final en TEST
test_loss, test_accuracy = model.evaluate(X_test, Y_test, verbose=0)
print(f"Precisión final en datos de PRUEBA: {test_accuracy*100:.2f}%")

## 6. Exportación del Modelo para Web App
Guardamos el modelo para consumirlo con TensorFlow.js.

In [ ]:
# Crear carpeta si no existe
if not os.path.exists('../modelo_final/'):
    os.makedirs('../modelo_final/')

ruta_modelo = '../modelo_final/modelo_cnn_sign_language.keras'
model.save(ruta_modelo)
print(f"¡Modelo guardado con éxito en: {ruta_modelo}!")